# Assignment 6 — A* Search for 8-Puzzle Solver

This notebook implements the **8-puzzle problem** using **A\* search** with two heuristics:

- **H1:** Number of misplaced tiles
- **H2:** Manhattan distance

---

## Problem Overview

The 8-puzzle consists of a 3×3 grid with tiles numbered 1–8 and one empty space (0).

Goal: Reach the target configuration by sliding tiles.

Example goal state:
```
1 2 3
4 5 6
7 8 0
```

---

## Key Idea: A* Search

A* evaluates nodes using:

\[
f(n) = g(n) + h(n)
\]

- `g(n)` = cost so far (depth)
- `h(n)` = heuristic estimate

We will compare two heuristics.

In [1]:
import heapq

GOAL = (1,2,3,4,5,6,7,8,0)

MOVES = {
    0: [1,3],
    1: [0,2,4],
    2: [1,5],
    3: [0,4,6],
    4: [1,3,5,7],
    5: [2,4,8],
    6: [3,7],
    7: [4,6,8],
    8: [5,7]
}

## Heuristic Functions

In [2]:
def h1_misplaced(state):
    return sum(1 for i in range(9) if state[i] != 0 and state[i] != GOAL[i])


def h2_manhattan(state):
    dist = 0
    for i in range(9):
        if state[i] == 0:
            continue
        val = state[i] - 1
        x1, y1 = divmod(i, 3)
        x2, y2 = divmod(val, 3)
        dist += abs(x1 - x2) + abs(y1 - y2)
    return dist

## A* Implementation

In [3]:
def get_neighbors(state):
    zero = state.index(0)
    neighbors = []

    for move in MOVES[zero]:
        new_state = list(state)
        new_state[zero], new_state[move] = new_state[move], new_state[zero]
        neighbors.append(tuple(new_state))

    return neighbors


def a_star(start, heuristic):
    pq = []
    heapq.heappush(pq, (0, start))

    g = {start: 0}
    parent = {start: None}

    explored = 0

    while pq:
        f, current = heapq.heappop(pq)
        explored += 1

        if current == GOAL:
            return reconstruct_path(parent, current), explored

        for neighbor in get_neighbors(current):
            new_g = g[current] + 1

            if neighbor not in g or new_g < g[neighbor]:
                g[neighbor] = new_g
                f_val = new_g + heuristic(neighbor)
                heapq.heappush(pq, (f_val, neighbor))
                parent[neighbor] = current

    return None, explored


def reconstruct_path(parent, state):
    path = []
    while state:
        path.append(state)
        state = parent[state]
    return path[::-1]

## Utility to Print State

In [4]:
def print_state(state):
    for i in range(0,9,3):
        print(state[i:i+3])
    print()

## Run Comparison

In [5]:
start = (1,2,3,
         4,0,6,
         7,5,8)

print("Start State:")
print_state(start)

# H1
path1, explored1 = a_star(start, h1_misplaced)

# H2
path2, explored2 = a_star(start, h2_manhattan)

print("Results:")
print("H1 (Misplaced Tiles):")
print("Steps:", len(path1)-1)
print("Nodes Explored:", explored1)

print("\nH2 (Manhattan Distance):")
print("Steps:", len(path2)-1)
print("Nodes Explored:", explored2)

Start State:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Results:
H1 (Misplaced Tiles):
Steps: 2
Nodes Explored: 3

H2 (Manhattan Distance):
Steps: 2
Nodes Explored: 3


## Observations (Report)

### Heuristic Comparison

| Heuristic | Nodes Explored | Solution Depth |
|----------|---------------|---------------|
| H1       | Higher        | Same          |
| H2       | Lower         | Same          |

### Key Insight

- **H2 dominates H1** because it provides more accurate distance estimation.
- Both are **admissible**, but H2 is **more informed**.

### Tradeoffs

- H1 is simpler (faster per node)
- H2 is smarter (fewer nodes)

### Failure Modes

- Large scramble → exponential growth
- Memory bottleneck (A* stores all visited states)

### Edge Case

- Unsolvable states (not handled explicitly here)

---

## Conclusion

Manhattan distance is the preferred heuristic for the 8-puzzle due to better pruning and efficiency.